# Предварительно обученные модели для векторизации текстовых данных

Здесь не будет библиотеки Spacy, но будут ее составляющие!

In [1]:
! pip install pymorphy3

## Загрузка данных

Будем использовать [коллекцию постов в Telegram-канале ПМиФИ (до 27.09.2023) ](https://drive.google.com/file/d/1yZY02dhISMzrtza_gdPRCjuzsgdOmMWz/view?usp=sharing).

In [2]:
import json

In [3]:
with open('pmifi-tg-27-09-2023.json', 'r', encoding='utf-8') as f:
  raw_json = json.load(f)

In [4]:
raw_json['messages'][:3]

[{'id': 1,
  'type': 'service',
  'date': '2022-03-10T11:33:46',
  'date_unixtime': '1646890426',
  'actor': 'ПМиФИ ОмГТУ',
  'actor_id': 'channel1687099166',
  'action': 'create_channel',
  'title': 'ПМиФИ',
  'text': '',
  'text_entities': []},
 {'id': 2,
  'type': 'message',
  'date': '2022-03-10T11:44:36',
  'date_unixtime': '1646891076',
  'edited': '2022-03-10T12:03:15',
  'edited_unixtime': '1646892195',
  'from': 'ПМиФИ ОмГТУ',
  'from_id': 'channel1687099166',
  'text': 'Всем добрый день! Будем заселять новые земли!',
  'text_entities': [{'type': 'plain',
    'text': 'Всем добрый день! Будем заселять новые земли!'}]},
 {'id': 3,
  'type': 'service',
  'date': '2022-03-10T11:47:06',
  'date_unixtime': '1646891226',
  'actor': 'ПМиФИ ОмГТУ',
  'actor_id': 'channel1687099166',
  'action': 'edit_group_title',
  'title': 'ПМиФИ ОмГТУ',
  'text': '',
  'text_entities': []}]

In [5]:
raw_messages = [' '.join(map(lambda x: x['text'], item['text_entities'])) for item in raw_json['messages'] if item['type'] == 'message']
raw_messages = list(filter(lambda x: len(x) > 0, raw_messages))

In [6]:
raw_messages[:3]

['Всем добрый день! Будем заселять новые земли!',
 'Привет!\n\nВ этом чате мы, кафедра ПМиФИ, делимся полезной информацией, отвечаем на вопросы и приглашаем на наши мероприятия!\n\nВ чате нельзя оскорблять других участников и использовать ненормативную лексику.\n\nПо вопросам поступления обращаться к Виктории Мунько, почта  vkreydunova@mail.ru , тел.  8-923-694-72-55 .',
 'Теперь вопросы и комментарии можно писать в обсуждениях к сообщению.\n\nА уже совсем скоро мы расскажем про различие направлений МО и ФИТ 🔥']

## Предварительная обработка данных

Удалим эмоджи, почты и телефоны с помощью регулярных выражений

In [7]:
import re

In [8]:
emoj = re.compile('['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F6FF'
        u'\U0001F1E0-\U0001F1FF'
        u'\U00002500-\U00002BEF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        u'\U0001f926-\U0001f937'
        u'\U00010000-\U0010ffff'
        u'\u2640-\u2642'
        u'\u2600-\u2B55'
        u'\u200d'
        u'\u23cf'
        u'\u23e9'
        u'\u231a'
        u'\ufe0f'
        u'\u3030'']+||(\S+@\S+\.\S+)||([+]*[(]{0,1}[0-9]{1,4}[)]{0,1}[-\s\./0-9]*)', re.UNICODE)

In [9]:
messages = [re.sub(emoj, '', msg) for msg in raw_messages]
messages[:3]

['Всем добрый день! Будем заселять новые земли!',
 'Привет!\n\nВ этом чате мы, кафедра ПМиФИ, делимся полезной информацией, отвечаем на вопросы и приглашаем на наши мероприятия!\n\nВ чате нельзя оскорблять других участников и использовать ненормативную лексику.\n\nПо вопросам поступления обращаться к Виктории Мунько, почта   , тел.  ',
 'Теперь вопросы и комментарии можно писать в обсуждениях к сообщению.\n\nА уже совсем скоро мы расскажем про различие направлений МО и ФИТ ']

Продолжаем предварительную обработку

In [10]:
messages = [re.sub('[^\s^\w]+', '', msg) for msg in messages]
messages[:3]

['Всем добрый день Будем заселять новые земли',
 'Привет\n\nВ этом чате мы кафедра ПМиФИ делимся полезной информацией отвечаем на вопросы и приглашаем на наши мероприятия\n\nВ чате нельзя оскорблять других участников и использовать ненормативную лексику\n\nПо вопросам поступления обращаться к Виктории Мунько почта    тел  ',
 'Теперь вопросы и комментарии можно писать в обсуждениях к сообщению\n\nА уже совсем скоро мы расскажем про различие направлений МО и ФИТ ']

In [11]:
messages = [re.sub('\s+', ' ', msg).strip() for msg in messages]
messages[:3]

['Всем добрый день Будем заселять новые земли',
 'Привет В этом чате мы кафедра ПМиФИ делимся полезной информацией отвечаем на вопросы и приглашаем на наши мероприятия В чате нельзя оскорблять других участников и использовать ненормативную лексику По вопросам поступления обращаться к Виктории Мунько почта тел',
 'Теперь вопросы и комментарии можно писать в обсуждениях к сообщению А уже совсем скоро мы расскажем про различие направлений МО и ФИТ']

Нормализизуем и удаляем стоп-слова

In [12]:
from pymorphy3 import MorphAnalyzer
morph = MorphAnalyzer()

In [13]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize

In [15]:
messages = [[morph.normal_forms(w)[0] for w in word_tokenize(msg)] for msg in messages]

In [16]:
messages[0]

['весь', 'добрый', 'день', 'быть', 'заселять', 'новый', 'земля']

In [17]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords = stopwords.words('russian')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
messages = [[w for w in msg if w not in stopwords] for msg in messages]
messages[0]

['весь', 'добрый', 'день', 'заселять', 'новый', 'земля']

## Обучение моделей Word2Vec с помощью Gensim

[Gensim](https://radimrehurek.com/gensim/) - библиотека, которая изначально разрабатывалась для решения задач тематического моделирования. Однако с помощью нее удобно обучать модели Word2Vec на собственных данных. При обучении используется алгоритм CBoW.

In [19]:
from gensim.models import Word2Vec

In [20]:
w2v = Word2Vec(sentences=messages, min_count=1, vector_size=64, epochs=50)

Векторные представления слов в Gensim имеют тип KeyedVectors. Это важно понимать, поскольку в интернете можно найти много векторных представлений в этом формате. Все они могут быть загружены и использованы с помощью Gensim. Далее мы это увидим.

In [21]:
w2v.wv

In [22]:
w2v.wv['кафедра'], w2v.wv['кафедра'].shape

(array([ 0.73101026, -0.6297193 ,  0.60533327,  0.30236912, -0.21029195,
        -0.83699816,  0.02110135, -0.00942422, -0.48491606,  0.02925689,
         1.1594096 , -0.33694804, -0.39040506, -0.4233467 , -0.49917632,
         0.4790906 , -0.75102645,  0.20526253, -0.12417392,  0.6829561 ,
         0.79317445,  0.9056127 ,  0.8554256 , -0.86894095, -0.14438401,
         0.6924468 , -0.43902066,  0.27793694,  0.5334049 , -0.14932868,
         0.02420786,  0.57747936, -0.4854701 , -0.40924814, -0.28028816,
        -0.36837158,  0.25656402,  0.36249176,  0.3594085 , -0.05671677,
         0.39213973,  0.53505516, -0.7088474 , -0.26999694,  0.6702818 ,
        -0.10364173, -0.47807145, -0.41532204, -0.2688912 ,  1.0752767 ,
        -0.0051323 , -0.02922777, -0.00521556,  0.87906927,  0.1323994 ,
        -0.12003546,  0.6618241 , -1.0737616 , -0.35908595,  0.11927351,
        -0.1183393 , -0.27652425,  0.4782811 ,  0.3832619 ], dtype=float32),
 (64,))

А вот здесь начинается магия!

In [23]:
w2v.wv.most_similar('кафедра', topn=3)

[('преподаватель', 0.9810979962348938),
 ('наш', 0.9804729223251343),
 ('пмифь', 0.9690815806388855)]

In [24]:
w2v.wv.most_similar('конференция', topn=5)

[('молодёжный', 0.9894673824310303),
 ('xii', 0.9782147407531738),
 ('международный', 0.9780377149581909),
 ('xiii', 0.9664879441261292),
 ('открытие', 0.9648135900497437)]

Можем сохранить нашу модель Word2Vec, чтобы использовать ее в будущем

In [25]:
w2v.save('myw2v.bin')

In [26]:
w2v2 = Word2Vec.load('myw2v.bin')
w2v2.wv.most_similar('конференция', topn=5)

[('молодёжный', 0.9894673824310303),
 ('xii', 0.9782147407531738),
 ('международный', 0.9780377149581909),
 ('xiii', 0.9664879441261292),
 ('открытие', 0.9648135900497437)]

## Предварительно обученные модели RusVectores

Для русского и украинского языка существууют предварительно обученные модели [RusVectores](https://rusvectores.org/ru/models/)

Самую известную модель с векторами размерности 300, обученную на уже известном вам корпусе RuCorpora, можно загрузить используя API Gensim. Другие модели можно скачать с сайта и загрузить с диска в Gensim, как было показано выше.

С помощью API Gensim можно скачать самые разные модели

In [27]:
from gensim.downloader import load, info

In [28]:
list(info()['models'].keys())

['fasttext-wiki-news-subwords-300',
 'conceptnet-numberbatch-17-06-300',
 'word2vec-ruscorpora-300',
 'word2vec-google-news-300',
 'glove-wiki-gigaword-50',
 'glove-wiki-gigaword-100',
 'glove-wiki-gigaword-200',
 'glove-wiki-gigaword-300',
 'glove-twitter-25',
 'glove-twitter-50',
 'glove-twitter-100',
 'glove-twitter-200',
 '__testing_word2vec-matrix-synopsis']

In [29]:
rv = load('word2vec-ruscorpora-300')

In [30]:
rv

Обратите внимание, что тип данных - KeyedVectors. Ранее мы оборачивали этот тип в объект Word2Vec.

In [31]:
rv.most_similar('кафедра_NOUN')

[('доцент_NOUN', 0.6663437485694885),
 ('декан_NOUN', 0.6478123664855957),
 ('профессор_NOUN', 0.6361485123634338),
 ('факультет_NOUN', 0.6278356909751892),
 ('университет_NOUN', 0.6136446595191956),
 ('преподаватель_NOUN', 0.6078613996505737),
 ('ректор_NOUN', 0.6058201789855957),
 ('мирэа_NOUN', 0.5775336027145386),
 ('тихоновский::гуманитарный_ADJ', 0.5705974102020264),
 ('адъюнкт-профессор_NOUN', 0.5699172616004944)]

In [62]:
rv['кафедра_NOUN'].shape

(300,)

In [32]:
import numpy as np
rv.similar_by_vector(np.zeros(300))

[('пахтать_VERB', 0.0),
 ('свидетель::платайс_NOUN', 0.0),
 ('амфиктион_NOUN', 0.0),
 ('эйс_NOUN', 0.0),
 ('германос_NOUN', 0.0),
 ('етоты_NOUN', 0.0),
 ('козырева_NOUN', 0.0),
 ('микроген_NOUN', 0.0),
 ('натюрель_NOUN', 0.0),
 ('ахмедова::щель_NOUN', 0.0)]

## Предварительно обученные модели FastText

FastText - это способ организации векторного пространства для текстовых данных, при котором мы сопоставляем вектора не словам, а n-граммам символов, из которых они состоят. Векторное представление слова строится как среднее значение векторов входящих в него n-грамм.

Для обучения и использования моделей с использованием данного подхода существует библиотека [fasttext](https://fasttext.cc/)

In [33]:
! pip install fasttext

Как и Gensim библиотека fasttext позволяет обучать собственные модели Word2Vec, однако в данном случае мы посмотрим на предварительно обученную модель для русского языка

In [38]:
from fasttext.util import download_model
import fasttext

Модель для русского языка весит 4 ГБ.

In [35]:
#ft_pre = fasttext.load_model(download_model('ru'))

In [37]:
a = ' '.join([' '.join(msg) for msg in messages])
with open('all_messages_text.txt', 'w') as f:
  f.write(a)

In [57]:
ft = fasttext.train_unsupervised('all_messages_text.txt', model='skipgram', dim=64)

In [58]:
ft.get_nearest_neighbors('математика', k=4)

[(0.9999982118606567, 'математик'),
 (0.9999945163726807, 'математический'),
 (0.9999932050704956, 'информатика'),
 (0.9999926686286926, 'материал')]

In [59]:
ft.get_word_vector('кафедра').shape

(64,)

In [60]:
ft.save_model('ft.bin')

## Предварительно обученная модель Navec и проект Natasha

В 2020 году [Лаборатория анализа данных Александра Кукушкина](https://lab.alexkuk.ru/) представила проект с открытым исходным кодом [Natasha](https://github.com/natasha/natasha).

Проект объединил в себе современные инструменты для обработки русского языка (токенизатор [Razdel](https://github.com/natasha/razdel), векторизатор [Navec](https://github.com/natasha/navec), размеченный для решения морфологических задач набор данных [Nerus](https://github.com/natasha/nerus) и др.)

**Все эти инструменты используются внутри моделей для русского языка фреймворка Spacy!**

Поэтому их можно использовать отдельно друг от друга. Razdel работает для русского языка лучше, чем стандартный токенизатор из NLTK. Рекомендуется использовать его

In [63]:
! pip install razdel

In [64]:
from razdel import tokenize

In [68]:
tokens = list(tokenize('Где-то на высоте 5 м кубических, площади 3м 2'))
tokens

[Substring(0, 6, 'Где-то'),
 Substring(7, 9, 'на'),
 Substring(10, 16, 'высоте'),
 Substring(17, 18, '5'),
 Substring(19, 20, 'м'),
 Substring(21, 31, 'кубических'),
 Substring(31, 32, ','),
 Substring(33, 40, 'площади'),
 Substring(41, 42, '3'),
 Substring(42, 43, 'м'),
 Substring(44, 45, '2')]

In [71]:
tokens[0]

Substring(0, 6, 'Где-то')

In [72]:
tokens[0].text

'Где-то'

Займемся векторизацией

In [73]:
! pip install navec

In [74]:
! wget https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar

--2023-09-27 18:17:55--  https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53012480 (51M) [application/x-tar]
Saving to: ‘navec_hudlit_v1_12B_500K_300d_100q.tar’

navec_hudlit_v1_12B 100%[===================>]  50.56M  15.9MB/s    in 3.5s    

2023-09-27 18:18:00 (14.4 MB/s) - ‘navec_hudlit_v1_12B_500K_300d_100q.tar’ saved [53012480/53012480]



In [75]:
from navec import Navec

In [76]:
navec = Navec.load('navec_hudlit_v1_12B_500K_300d_100q.tar')

In [78]:
navec['кафедра'].shape

(300,)

In [84]:
'мы' in navec, 'пмифи' in navec

(True, False)

Предусмотрен вектор для неизвестных слов

In [86]:
navec['<unk>'].shape

(300,)

Также предусмотрен вектор из нулей для использования при заполнении последовательностей (про это мы еще поговорим позже)

In [88]:
navec['<pad>'].shape, sum(navec['<pad>'])

((300,), 0.0)

Определяем семантическую близость слов (косинус между векторами)

In [97]:
navec.sim('профессор', 'ректор')

0.6199871

Теперь вы можете использовать при решении своих задач (в том числе с применением рекуррентных нейронных сетей) предварительно обученные векторные представления.

Далее в курсе NLP мы еще вернемся к векторизации текстов, будем говорить уже про векторизацию предложений. Но сначала нужно будет познакомиться с такой архитектурой нейронных сетей как трансформер.